# Train a tabular or text model for Local AI Lab (scikit-learn → ONNX)

This notebook trains a model **only on the training rows** of a Model Arena dataset, exports it to ONNX with the
metadata Local AI Lab understands (labels, feature columns, dataset and split hashes), and gives you a file to upload
under **Model Arena → Models**. Because the split hash matches, the Arena knows the model never saw the validation or
test rows, and compares it fairly with the built-in algorithms.

> **Privacy first.** Google Colab runs on Google's cloud. Only upload data you are allowed to process there –
> **never confidential documents, patient, customer or employee data** unless your organisation approved it.
> The same notebook runs on your own computer with Jupyter (`pip install notebook`, then `jupyter notebook`), which
> keeps everything offline. Using Colab is subject to Google's terms; the libraries used here are open source
> (licences listed at the end).

**You need from Local AI Lab:** the CSV you uploaded, and *Model Arena → your dataset → Download split* (`split.csv`).

In [ ]:
# 1. Install (Colab already has pandas / scikit-learn; skl2onnx converts the pipeline to ONNX)
%pip install -q "skl2onnx>=1.17" "onnx>=1.16" "onnxruntime>=1.18"

## 2. Upload the data
In Colab a file picker opens. On your own computer, set the two paths instead.

In [ ]:
import io, os, json
CSV_PATH = "data.csv"        # your dataset (same file you uploaded to Local AI Lab)
SPLIT_PATH = "split.csv"     # Model Arena → dataset → Download split
try:
    from google.colab import files  # only in Colab
    up = files.upload()
    names = list(up)
    CSV_PATH = next(n for n in names if "split" not in n.lower())
    SPLIT_PATH = next(n for n in names if "split" in n.lower())
except ImportError:
    pass
print(CSV_PATH, SPLIT_PATH)

## 3. Settings – edit these

In [ ]:
TARGET = "churn"            # the target column you chose in Local AI Lab
TASK = "classification"      # or "regression"
TEXT_COLUMN = None           # e.g. "ticket" for a text dataset (one long text column), else None
FEATURE_COLUMNS = None       # None = every column except the target / ignored ones below
IGNORE = ["customer_id"]     # ID-like columns (Local AI Lab ignores them too)
MODEL = "random_forest"      # tabular: random_forest | gradient_boosting | logistic_regression ; text: logistic_regression

In [ ]:
import csv
import numpy as np
import pandas as pd

with open(CSV_PATH, "rb") as f:
    raw = f.read()
text = raw.decode("utf-8-sig", errors="replace")
sep = csv.Sniffer().sniff(text[:65536], delimiters=",;\t|").delimiter
df = pd.read_csv(io.StringIO(text), sep=sep, low_memory=False, skipinitialspace=True)
df.columns = [str(c).strip() for c in df.columns]

with open(SPLIT_PATH, encoding="utf-8") as f:
    header = f.readline()
meta = dict(kv.split("=", 1) for kv in header.lstrip("# ").split() if "=" in kv)
split = pd.read_csv(SPLIT_PATH, skiprows=1)  # first line: hashes
DATASET_SHA256, SPLIT_SHA256 = meta.get("dataset_sha256"), meta.get("split_sha256")
print("dataset", DATASET_SHA256[:12], "split", SPLIT_SHA256[:12])
print(split["split"].value_counts())

train_rows = split.loc[split["split"] == "train", "source_row"].astype(int).to_numpy()
train = df.iloc[train_rows].copy()
print(f"training on {len(train)} of {len(df)} rows – validation and test rows are NOT used")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

y = train[TARGET]
if TASK == "classification":
    y = y.map(lambda v: str(int(v)) if isinstance(v, float) and float(v).is_integer() else str(v).strip())
else:
    y = pd.to_numeric(y, errors="coerce").astype(float)

if TEXT_COLUMN:
    features = [TEXT_COLUMN]
    X = train[TEXT_COLUMN].fillna("").astype(str).to_numpy().reshape(-1, 1)
    est = LogisticRegression(max_iter=2000) if TASK == "classification" else Ridge()
    pipe = Pipeline([("tfidf", ColumnTransformer([("t", TfidfVectorizer(max_features=50000, ngram_range=(1, 1)), 0)])),
                     ("model", est)])
else:
    features = FEATURE_COLUMNS or [c for c in df.columns if c != TARGET and c not in IGNORE]
    num = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in features if c not in num]
    X = train[features].copy()
    for c in num:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype(np.float32)
    for c in cat:  # missing text values become "" – exactly what Local AI Lab feeds the model
        X[c] = X[c].fillna("").astype(str)
    pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    if TASK == "classification":
        est = {"random_forest": RandomForestClassifier(200, random_state=0),
               "gradient_boosting": GradientBoostingClassifier(random_state=0),
               "logistic_regression": LogisticRegression(max_iter=2000)}[MODEL]
    else:
        est = {"random_forest": RandomForestRegressor(200, random_state=0),
               "gradient_boosting": GradientBoostingRegressor(random_state=0),
               "logistic_regression": Ridge()}[MODEL]
    pipe = Pipeline([("pre", pre), ("model", est)])

pipe.fit(X, y)
print("trained", type(est).__name__, "on", len(features), "feature column(s)")

## 4. Export to ONNX with Local AI Lab metadata

In [ ]:
import onnx
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

if TEXT_COLUMN:
    initial_types = [(TEXT_COLUMN, StringTensorType([None, 1]))]
else:
    initial_types = [(c, FloatTensorType([None, 1]) if c in num else StringTensorType([None, 1])) for c in features]
options = {id(pipe): {"zipmap": False}} if TASK == "classification" else {}
if TEXT_COLUMN:  # "C" locale works on every OS / container (en_US may be missing on minimal Linux images)
    options[TfidfVectorizer] = {"locale": "C"}
onx = to_onnx(pipe, initial_types=initial_types, options=options or None, target_opset=17)

labels = [str(c) for c in getattr(pipe, "classes_", [])]
meta = {"kind": "text" if TEXT_COLUMN else "tabular", "task": TASK,
        "feature_columns": json.dumps(features), "dataset_sha256": DATASET_SHA256, "split_sha256": SPLIT_SHA256,
        "trained_on": "train", "framework": "scikit-learn + skl2onnx",
        "notebook": "local-ai-lab/notebooks/train_tabular_text_sklearn.ipynb"}
if labels:
    meta["labels"] = json.dumps(labels)
for k, v in meta.items():
    if v is not None:
        p = onx.metadata_props.add()
        p.key, p.value = f"local_ai_lab.{k}", str(v)
OUT = "model.onnx"
onnx.save(onx, OUT)
print("saved", OUT, f"{os.path.getsize(OUT) / 1024:.0f} KB", "labels:", labels)

In [ ]:
# 5. Sanity check with ONNX Runtime on a few training rows (the Arena does the real evaluation)
import onnxruntime as ort
sess = ort.InferenceSession(OUT, providers=["CPUExecutionProvider"])
sample = X[:5] if TEXT_COLUMN else X.iloc[:5]
if TEXT_COLUMN:
    feeds = {TEXT_COLUMN: np.asarray(sample, dtype=object).reshape(-1, 1)}
else:
    feeds = {c: (sample[c].to_numpy(np.float32) if c in num else sample[c].astype(str).to_numpy(object)).reshape(-1, 1)
             for c in features}
print(sess.run(None, feeds)[0][:5])

In [ ]:
# 6. Download the model
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    print("model written to", os.path.abspath(OUT))

## 7. Use it in Local AI Lab
**Model Arena → Models → Upload model** → pick `model.onnx`, data type = tabular/text, task = the same.
Labels and feature columns are read from the file. "Training data" shows *trained only on this dataset's training
split* because the split hash matches. Then start a comparison on the same dataset.

**Licences:** scikit-learn (BSD-3-Clause), skl2onnx / ONNX (Apache-2.0), ONNX Runtime (MIT), pandas / NumPy (BSD).
Your data and the trained model are yours; make sure you have the right to use the data for training.